In [1]:
import sys
from pathlib import Path
import json
import pickle

ROOT = Path.cwd().resolve()
if (ROOT / 'src').exists():
    repo_root = ROOT
elif (ROOT.parent / 'src').exists():
    repo_root = ROOT.parent
else:
    repo_root = ROOT.parent.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f'Using repo root: {repo_root}')

Using repo root: D:\git projects\certified-attribution-medical-imaging


In [ ]:
# Configuration
import torch
from src.certify.eval.faithfulness import FaithfulnessEvaluator
from src.datasets.grid_dataset import GridDataset
from src.models.grid_multihead import GridMultiHead

dataset_name = 'isic_grid'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Paths to grid certification results and checkpoints
cert_results_dir = repo_root / 'outputs/bulk_certifcation/grid/isic/resnet18'
output_base = repo_root / 'notebooks/output/eval/faithfullness/grid/isic'
checkpoint_base = repo_root / 'outputs/checkpoints/isic'

grid_pt = repo_root / 'data/raw/grid/isic/val/grid.pt'
checkpoint_path = checkpoint_base / 'resnet18/final_model.pt'

print(f'Cert results dir: {cert_results_dir}')
print(f'Output base: {output_base}')
print(f'Checkpoint: {checkpoint_path}')
print(f'Grid payload: {grid_pt}')

Device: cpu
Cert results dir: D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic
Output base: D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\faithfullness\isic
Checkpoint base: D:\git projects\certified-attribution-medical-imaging\notebooks\output\checkpoints\isic


In [ ]:
# Find grid certification results (pick latest .pkl)
import glob
cert_files = sorted(glob.glob(str(cert_results_dir / '**/*.pkl'), recursive=True))
print(f'Found {len(cert_files)} grid certification result(s):')
for f in cert_files:
    print('  ', Path(f).name)

if not cert_files:
    raise FileNotFoundError(f'No certification results found in {cert_results_dir}')

cert_pkl = cert_files[-1]
print(f'Using: {cert_pkl}')

Found 2 certification result(s):
   results_20251229_163021.pkl
   results_partial.pkl
Using: D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic\results_20251229_163021.pkl


In [ ]:
# Load certification results to see available models
with open(cert_pkl, 'rb') as f:
    cert_results = pickle.load(f)

models_in_results = [m for m in cert_results.keys() if cert_results[m]]
print('Models with results:', models_in_results)
for model_name in models_in_results:
    methods = list(cert_results[model_name].keys())
    print(f'  {model_name}: {methods}')

Models with results: ['densenet121']
  densenet121: ['IntegratedGradients', 'GradCAM', 'RISE', 'Occlusion', 'LRP']


In [ ]:
# Load grid dataset (matches certification)
from src.datasets.grid_dataset import GridDataset

dataset = GridDataset(Path(grid_pt))
print(f'Grid dataset size: {len(dataset)} | scale={dataset.scale} | target_cell={dataset.target_cell}')

Dataset size (val): 522


In [ ]:
# Helper: load multi-head grid model initialized from single-head checkpoint
def load_grid_model(checkpoint: Path, num_heads: int, num_classes: int, device: str):
    model = GridMultiHead('resnet18', num_classes=num_classes, num_heads=num_heads, pretrained=(not checkpoint.exists()))
    if checkpoint.exists():
        state = torch.load(checkpoint, map_location=device)
        if isinstance(state, dict):
            if 'model_state_dict' in state:
                state = state['model_state_dict']
            elif 'state_dict' in state:
                state = state['state_dict']
        state = {k.replace('module.', ''): v for k, v in state.items()}
        model_dict = model.state_dict()
        state = {k: v for k, v in state.items() if k in model_dict}
        model_dict.update(state)
        model.load_state_dict(model_dict, strict=False)
        model.duplicate_head_weights(0)
        print(f'Loaded checkpoint into grid model: {checkpoint}')
    else:
        print(f'Checkpoint not found, using ImageNet-pretrained backbone: {checkpoint}')
    model.to(device)
    model.eval()
    return model

# Lightweight wrapper to avoid BaseEvaluator model loading
class GridFaithfulnessEvaluator(FaithfulnessEvaluator):
    def __init__(self, dataset_name: str, model_name: str, model: torch.nn.Module, device: str):
        self.dataset_name = dataset_name
        self.model_name = model_name
        self.device = device
        self.model = model

all_results = {}
num_heads = dataset.scale * dataset.scale
grid_model = load_grid_model(checkpoint_path, num_heads=num_heads, num_classes=8, device=device)

for model_name in models_in_results:
    print('='*60)
    print(f'Evaluating {model_name} (grid)')
    print('='*60)
    
    fa_output_dir = output_base / model_name
    fa_output_dir.mkdir(parents=True, exist_ok=True)
    
    evaluator = GridFaithfulnessEvaluator(
        dataset_name=dataset_name,
        model_name=model_name,
        model=grid_model,
        device=device,
    )
    
    fa_results = evaluator.evaluate_batch(
        cert_results_pkl=Path(cert_pkl),
        dataset=dataset,
        output_dir=fa_output_dir,
        deletion_steps=5,
    )
    
    evaluator.save_results_json(fa_results, fa_output_dir / 'faithfulness_results.json')
    evaluator.plot_results(fa_results, fa_output_dir / 'figures')
    evaluator.plot_deletion_confidence_curves(
        cert_results_pkl=Path(cert_pkl),
        dataset=dataset,
        output_dir=fa_output_dir / 'figures',
        deletion_steps=4,
    )
    
    all_results[model_name] = fa_results
    print(f'Completed {model_name}: methods={list(fa_results.keys())}')

Evaluating densenet121
  Loading checkpoint: D:\git projects\certified-attribution-medical-imaging\notebooks\output\checkpoints\isic\densenet121\final_model.pt


d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



  Evaluating GradCAM...



  Evaluating IntegratedGradients...



  Evaluating LRP...



  Evaluating Occlusion...



  Evaluating RISE...


  ✓ Saved results to D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\faithfullness\isic\densenet121\faithfulness_results.json
  ✓ Saved faithfulness figure to D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\faithfullness\isic\densenet121\figures\faithfulness_comparison.png
  ✓ Saved deletion-step confidence data to D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\faithfullness\isic\densenet121\figures\faithfulness_confidence_curves_data.json
  ✓ Saved deletion-step confidence curves to D:\git projects\certified-attribution-medical-imaging\notebooks\output\eval\faithfullness\isic\densenet121\figures\faithfulness_confidence_curves.png
Completed densenet121: methods=['GradCAM', 'IntegratedGradients', 'LRP', 'Occlusion', 'RISE']


In [7]:
# Display summary tables
import pandas as pd

for model_name, model_results in all_results.items():
    print('' + '='*60)
    print(model_name.upper())
    print('='*60)
    for method_name, k_dict in model_results.items():
        print(f'{method_name}:')
        rows = []
        for k in sorted(k_dict.keys()):
            metrics = k_dict[k]
            rows.append({
                'K%': k,
                'Mean AUC': f
,
                'Std AUC': f
,
                'Num images': metrics.get('num_images', 0),
                'Mean baseline conf': f
,
            })
        if rows:
            df = pd.DataFrame(rows)
            print(df.to_string(index=False))
        else:
            print('  (no results)')

DENSENET121
GradCAM:
 K%                                                                                                                                                  Mean AUC                                                                                                                                                   Std AUC  Num images                                                                                                                                        Mean baseline conf
  5 <_io.BufferedReader name='D:\\git projects\\certified-attribution-medical-imaging\\notebooks\\output\\certifications\\isic\\results_20251229_163021.pkl'> <_io.BufferedReader name='D:\\git projects\\certified-attribution-medical-imaging\\notebooks\\output\\certifications\\isic\\results_20251229_163021.pkl'>           3 <_io.BufferedReader name='D:\\git projects\\certified-attribution-medical-imaging\\notebooks\\output\\certifications\\isic\\results_20251229_163021.pkl'>
 25 <_io.BufferedReader